# 2단계 모델 검증: 년도 기준 분할 (훈련 ~2023 / 테스트 2024~2025)

## 1. 라이브러리 임포트

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso

# 한글 폰트 설정
matplotlib.rc('font', family='Malgun Gothic')
matplotlib.rc('axes', unicode_minus=False)

## 2. 데이터 불러오기 및 전처리

In [2]:
# 검단 포함 전체 신도시 데이터 불러오기
folder_path = r"C:\Users\SJ\Desktop\26-1\데이터마이닝\파주시아파트"
df = pd.read_csv(folder_path + r"\real_new_city.csv", encoding='utf-8-sig')

# 거래금액 전처리 (쉼표 제거 후 정수 변환)
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',', '').astype(int)

# m2당가격 파생변수 생성
df['m2당가격'] = df['거래금액(만원)'] / df['전용면적(㎡)']

print(f"전체 데이터: {len(df)}건")

전체 데이터: 126375건


## 3. 이상치 제거

In [3]:
# 전용면적 33제곱미터 미만 제거
df = df[df['전용면적(㎡)'] >= 33]

# m2당가격 z-score 기준 이상치 제거 (|z| > 2인 데이터 제거)
mean = df['m2당가격'].mean()
std = df['m2당가격'].std()
z_scores = (df['m2당가격'] - mean) / std
df = df[z_scores.abs() <= 2]

print(f"이상치 제거 후: {len(df)}건")

이상치 제거 후: 120646건


## 4. 년도 기준 훈련/테스트 분할

In [ ]:
# Top 5 독립변수
features = ['건축년도', '층',
            '지하철호선개수', '기차역까지의거리',
            '가장 가까운 지하철역까지의 거리', '가장 가까운 IC와의 거리',
            '발표후경과년수', 'CPI', '계약연도', '서울도심거리',
            '단지별_세대수', '도시별_세대수']

# 결측치 제거
df = df.dropna(subset=features + ['m2당가격'])

# 년도 기준 분할
df_train = df[df['계약연도'] <= 2023]
df_test = df[df['계약연도'] >= 2024]s

# 독립변수와 타겟 분리
train_input = df_train[features]
train_target = df_train['m2당가격']
test_input = df_test[features]
test_target = df_test['m2당가격']

print(f"훈련 데이터 (~2023): {len(df_train)}건")
print(f"테스트 데이터 (2024~2025): {len(df_test)}건")

훈련 데이터 (~2023): 100577건
테스트 데이터 (2024~2025): 20063건


## 5. Lasso 모델 학습 및 성능 평가

In [5]:
# 표준화
ss = StandardScaler()
ss.fit(train_input)
train_scaled = ss.transform(train_input)
test_scaled = ss.transform(test_input)

# Lasso 모델 학습
lasso = Lasso(alpha=0.001, max_iter=1000000)
lasso.fit(train_scaled, train_target)

print(f"Lasso 훈련 R²: {lasso.score(train_scaled, train_target):.4f}")
print(f"Lasso 테스트 R²: {lasso.score(test_scaled, test_target):.4f}")

Lasso 훈련 R²: 0.6750
Lasso 테스트 R²: 0.4874
